In [2]:
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
import os
import numpy as np
pd.set_option('display.max_columns', None)

In [12]:
df_reporte_emision= pd.read_csv('c:/data/VIDRENTADEV - NOVIEMBRE 2025_94910.csv', sep=',', encoding='latin-1', dtype=str, skiprows=7)
#df_reporte_emision= pd.read_csv('c:/data/VIDRENTADEV - OCTUBRE 2025_94909.csv', sep=',', encoding='latin-1', dtype=str, skiprows=7)


In [13]:
df_reporte_emision.columns = (df_reporte_emision.columns.str.strip()
                                                    .str.upper()  # opcional: todo en mayúsculas
                                                    .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)  # reemplazar estapacios por _
)

df_reporte_emision.drop(['FEC__INICIO_AX','FEC__FIN_AX', 'PRIMA_BRUTA_AX', 
                         'TIPO_DOC_EMISI_N_ORIGEN', 'N__DOC_EMISI_N_ORIGEN',
                         'TASA_CON_RECARGO', 'PRIMABRUTACAN', 'PRIMANETACAN', 
                         'PLACA','MOTOR', 'L_NEA_DE_TRAMA', 'MONEDA_1', 'TASA',                    
                         'IGV','INTERES_FINANCIAMIENTO', 'IGV_INTERES_FINANCIAMIENTO', 
                         'OTROS_CONCEPTOS', 'COMISION', 'PRIMA_BRUTA_AE', 
                         'MONTO_COMISI_N_CANAL'], axis=1, inplace=True)


In [14]:
df_reporte_emision.rename(columns={
    'NRO_LOTES_ANTERIORES': 'LOTES_ANTERIORES',  'C_DIGO_PRODUCTO': 'CODIGO_PRODUCTO',
    'C_D__CERTIFICADO_CANAL': 'CERTIFICADO_CANAL',  'TIPO_DOC__ID': 'TIPO_DOCUMENTO',
    'DOC__ID_': 'ID_DOCUMENTO',  'FEC__INICIO': 'FECHA_INICIO',
    'FEC__FIN': 'FECHA_FIN',  'LINEA_TRAMA_TEXTO_COMPLETO_': 'LINEA_TRAMA',
    'DESCRIPCI_N_DE_ESTADO': 'DESCRIPCION_ESTADO',  'C_DIGO_DE_RAMO': 'CODIGO_RAMO',
    'C_DIGO_DE_PRODUCTO_EN_CORE': 'CODIGO_PROD_CORE',  'DERECHO_DE_EMISI_N': 'DERECHO_EMISION',
    'NUMERO_CERTIFICADO_RIMAC': 'CERTIFICADO_RIMAC',  'TIPO_DOC_EMISI_N': 'TIPO_DOC_EMISION',
    'N__DOC_EMISI_N': 'NRO_DOC_EMISION',  'FECHA_DE_EMISI_N': 'FECHA_EMISION', 
    'NUMERO_DE_POLIZA_AE': 'NRO_POLIZA_AE', 'FEC__INICIO_AE': 'FECHA_INICIO_AE',  
    'FEC__FIN_AE': 'FECHA_FIN_AE'
}, inplace=True)

In [15]:

df_reporte_emision= df_reporte_emision.fillna({'LOTES_ANTERIORES':'', 'CODIGO_RAMO': '', 'CODIGO_PROD_CORE':'',
                                               'NRO_DOC_EMISION': '','TIPO_DOC_EMISION': '', 'CONTRATANTE':'', 
                                               'RESPONSABLE_DE_PAGO': '' })

In [16]:
df_reporte_emision['CERTIFICADO_CANAL']= df_reporte_emision['CERTIFICADO_CANAL'].str[1:]
df_reporte_emision['CERTIFICADO_RIMAC']= df_reporte_emision['CERTIFICADO_RIMAC'].str[1:]
df_reporte_emision['NUMERO_DE_POLIZA']= df_reporte_emision['NUMERO_DE_POLIZA'].str[1:]
df_reporte_emision["MONEDA"] = df_reporte_emision["MONEDA"].replace(r"^\s*$", None, regex=True)
df_reporte_emision["MONEDA"] = df_reporte_emision["MONEDA"].where(df_reporte_emision["MONEDA"].isin(["USD", "SOL"]), "SIN DATO")

In [17]:
df_reporte_emision['NRO_LOTE']= df_reporte_emision['NRO_LOTE'].fillna(0).astype('int')
df_reporte_emision['FECHA_CARGA_LOTE'] = pd.to_datetime(df_reporte_emision['FECHA_CARGA_LOTE'], format="%d/%m/%Y %H:%M:%S", errors='coerce').dt.date
df_reporte_emision["PRODUCTO"] = df_reporte_emision["PRODUCTO"].astype(str).str.upper()
df_reporte_emision["NOMBRE_DE_PLAN"] = df_reporte_emision["NOMBRE_DE_PLAN"].astype(str).str.upper()
df_reporte_emision["TIPO_MOVIMIENTO"] = df_reporte_emision["TIPO_MOVIMIENTO"].astype(str).str.upper()
df_reporte_emision['FECHA_NACIMIENTO'] = pd.to_datetime(df_reporte_emision['FECHA_NACIMIENTO'], format="%d/%m/%Y", errors='coerce').dt.date
df_reporte_emision['FECHA_INICIO'] = pd.to_datetime(df_reporte_emision['FECHA_INICIO'], format="%d/%m/%Y", errors='coerce').dt.date
df_reporte_emision['FECHA_FIN'] = pd.to_datetime(df_reporte_emision['FECHA_FIN'], format="%d/%m/%Y", errors='coerce').dt.date
df_reporte_emision['SUMA_ASEGURADA'] = pd.to_numeric(df_reporte_emision['SUMA_ASEGURADA'], errors="coerce").astype('float64')
df_reporte_emision['PRIMA_NETA'] = pd.to_numeric(df_reporte_emision['PRIMA_NETA'], errors="coerce").astype('float64')
df_reporte_emision['DERECHO_EMISION'] = pd.to_numeric(df_reporte_emision['DERECHO_EMISION'], errors="coerce").astype('float64')
df_reporte_emision['PRIMA_BRUTA'] = pd.to_numeric(df_reporte_emision['PRIMA_BRUTA'], errors="coerce").astype('float64')
df_reporte_emision['FECHA_EMISION'] = pd.to_datetime(df_reporte_emision['FECHA_EMISION'], format='%d-%b-%y', errors='coerce').dt.date
df_reporte_emision['FECHA_INICIO_AE'] = pd.to_datetime(df_reporte_emision['FECHA_INICIO_AE'], format="%d/%m/%Y", errors='coerce').dt.date
df_reporte_emision['FECHA_FIN_AE'] = pd.to_datetime(df_reporte_emision['FECHA_FIN_AE'], format="%d/%m/%Y", errors='coerce').dt.date

In [18]:
df_reporte_emision.head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA_LOTE,CODIGO_PRODUCTO,PRODUCTO,PLAN,NOMBRE_DE_PLAN,CERTIFICADO_CANAL,TIPO_MOVIMIENTO,NOMBRE,APELLIDO_PATERNO,APELLIDO_MATERNO,FECHA_NACIMIENTO,TIPO_DOCUMENTO,ID_DOCUMENTO,FECHA_INICIO,FECHA_FIN,MONEDA,SUMA_ASEGURADA,NOMBRE_ARCHIVO,LINEA_TRAMA,ESTADO,DESCRIPCION_ESTADO,CODIGO_RAMO,CODIGO_PROD_CORE,PRIMA_NETA,DERECHO_EMISION,PRIMA_BRUTA,CERTIFICADO_RIMAC,NUMERO_DE_POLIZA,TIPO_DOC_EMISION,NRO_DOC_EMISION,FECHA_EMISION,CONTRATANTE,RESPONSABLE_DE_PAGO,ESTADO_DE_DOCUMENTO,IDEREGISTRO,NRO_POLIZA_AE,FECHA_INICIO_AE,FECHA_FIN_AE
0,868953,,2022-10-17,4492,CONTINENTAL VIDA RENTA C/DEVOLUCIÓN,184329,VIDA RENTA BBVA DEV (USD),00117794554002535166,RENOVACION,JEAN PIER,GIRON,AGUILAR,1996-12-30,2,75916242,2022-10-14,2022-11-14,USD,25000.0,20100130204_0257001_20221017_002.TXT,8220011779455400253516603390205 01U...,Terminado,Emitido en A/X,VIDA,0257,6.80,0.20,7.0,0092054845,0092054420,LV,1755725897,2025-11-19,JEAN PIER GIRON AGUILAR,,COB,554454702,0092054420,2021-12-14,2026-12-14
1,868953,,2022-10-17,4492,CONTINENTAL VIDA RENTA C/DEVOLUCIÓN,184329,VIDA RENTA BBVA DEV (USD),00110486884000883378,RENOVACION,FINA ANGELA,DIAZ,CHALE,1983-07-04,2,41818479,2022-10-14,2022-11-14,USD,25000.0,20100130204_0257001_20221017_002.TXT,8220011048688400088337804860104 01U...,Terminado,Emitido en A/X,VIDA,0257,10.19,0.31,10.5,0047576833,0047576387,LV,1755727305,2025-11-19,FINA ANGELA DIAZ CHALE,,COB,554454109,0047576387,2019-01-14,2024-01-14
2,868953,,2022-10-17,4492,CONTINENTAL VIDA RENTA C/DEVOLUCIÓN,184329,VIDA RENTA BBVA DEV (USD),00117794504001965310,RENOVACION,FATIMA IRIS,YLLANES,SEGURA,1990-03-19,2,46251241,2022-10-13,2022-11-13,USD,25000.0,20100130204_0257001_20221017_002.TXT,8220011779450400196531001840205 01U...,Terminado,Emitido en A/X,VIDA,0257,6.80,0.20,7.0,0080583866,0080583788,LV,1755726615,2025-11-19,FATIMA IRIS YLLANES SEGURA,,COB,554454365,0080583788,2021-02-13,2026-02-13


In [19]:
df_reporte_emision['TIPO_DOCUMENTO'].value_counts()

TIPO_DOCUMENTO
2    8789
4     125
6       4
Name: count, dtype: int64

In [20]:
df_reporte_emision['CERTIFICADO_RIMAC'].isna().sum()

np.int64(0)

In [21]:
df_reporte_emision[df_reporte_emision['NRO_DOC_EMISION'].isna()].head()

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA_LOTE,CODIGO_PRODUCTO,PRODUCTO,PLAN,NOMBRE_DE_PLAN,CERTIFICADO_CANAL,TIPO_MOVIMIENTO,NOMBRE,APELLIDO_PATERNO,APELLIDO_MATERNO,FECHA_NACIMIENTO,TIPO_DOCUMENTO,ID_DOCUMENTO,FECHA_INICIO,FECHA_FIN,MONEDA,SUMA_ASEGURADA,NOMBRE_ARCHIVO,LINEA_TRAMA,ESTADO,DESCRIPCION_ESTADO,CODIGO_RAMO,CODIGO_PROD_CORE,PRIMA_NETA,DERECHO_EMISION,PRIMA_BRUTA,CERTIFICADO_RIMAC,NUMERO_DE_POLIZA,TIPO_DOC_EMISION,NRO_DOC_EMISION,FECHA_EMISION,CONTRATANTE,RESPONSABLE_DE_PAGO,ESTADO_DE_DOCUMENTO,IDEREGISTRO,NRO_POLIZA_AE,FECHA_INICIO_AE,FECHA_FIN_AE


In [22]:
df_reporte_emision.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8918 entries, 0 to 8917
Data columns (total 40 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   NRO_LOTE             8918 non-null   int64  
 1   LOTES_ANTERIORES     8918 non-null   object 
 2   FECHA_CARGA_LOTE     8918 non-null   object 
 3   CODIGO_PRODUCTO      8918 non-null   object 
 4   PRODUCTO             8918 non-null   object 
 5   PLAN                 8918 non-null   object 
 6   NOMBRE_DE_PLAN       8918 non-null   object 
 7   CERTIFICADO_CANAL    8918 non-null   object 
 8   TIPO_MOVIMIENTO      8918 non-null   object 
 9   NOMBRE               8918 non-null   object 
 10  APELLIDO_PATERNO     8918 non-null   object 
 11  APELLIDO_MATERNO     8918 non-null   object 
 12  FECHA_NACIMIENTO     8918 non-null   object 
 13  TIPO_DOCUMENTO       8918 non-null   object 
 14  ID_DOCUMENTO         8918 non-null   object 
 15  FECHA_INICIO         8918 non-null   o